In [1]:
print("Installing libs")
!pip install -q transformers accelerate bitsandbytes torch datasets gradio

Installing libs
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.9 MB/s eta 0:00:00


In [2]:
 #hugging face llogin
 import os
 from huggingface_hub import notebook_login,login
 notebook_login()
 print("login sucessful")

login sucessful


In [3]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from datasets import load_dataset
import gradio as gr
from IPython.display import display,Markdown
import random

if torch.cuda.is_available():
  print(f"GPU detected")
  torch.set_default_device("cuda")
else:
  print("No GPU detected")
  torch.set_default_device("cpu")

GPU detected


In [4]:
def print_markdown(text):
  display(Markdown(text))

In [5]:
dataset_id="KrossKinetic/all_news_finance_jsonl"
print(f"Loading dataset {dataset_id}")
news_dataset=load_dataset(dataset_id,split="train")
print("dataset loaded successfully")

Loading dataset KrossKinetic/all_news_finance_jsonl


README.md:   0%|          | 0.00/214 [00:00<?, ?B/s]

all_news.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/5062 [00:00<?, ? examples/s]

dataset loaded successfully


In [6]:
news_dataset

Dataset({
    features: ['_id', 'main_domain', 'title', 'description', 'created_at'],
    num_rows: 5062
})

In [7]:
print(news_dataset.features)

{'_id': Value('string'), 'main_domain': Value('string'), 'title': Value('string'), 'description': Value('string'), 'created_at': Value('string')}


In [8]:
news_dataset.to_pandas()

,_id,main_domain,title,description,created_at
0,6453d70d358e80adbfc4cb2b,cnbc.com,"Dow drops 400 points, turns negative for the y...",Regional banks led the broader market lower as...,2023-05-04T16:01:46.448Z
1,6453cf909a78e3af538abe44,cointelegraph.com,Bitcoin drops with stocks as analyst warns of ...,Bitcoin dips as the U.S. banking crisis engulf...,2023-05-04T15:25:28.809Z
2,6453cb87ccab8508100df076,co.uk,Bitcoin Price Analysis: 29370 Tested After Su...,Bitcoin (BTC/USD) sought to add to recent gain...,2023-05-04T15:12:00.971Z
3,6453afd269f3c1643cf0a4f6,bitcoinist.com,"Bitcoin Is 75% To Halving, Here's How Past Cyc...",The current Bitcoin cycle is now 75% on the wa...,2023-05-04T13:10:51.220Z
4,645399d92471d73ea0976d27,seekingalpha.com,"Iron Mountain FFO of $0.71 beats by $0.03, rev...",Iron Mountain press release (IRM): Q1 FFO of...,2023-05-04T11:41:12.498Z
...,...,...,...,...,...
5057,631b16e628ebb650077c97ff,cointelegraph.com,"Ethereum Merge can trigger high volatility, Bi...",BitMEX CEO Alexander Höptner said that their t...,2022-09-09T10:32:52.512Z
5058,6317e6bc2e929627be3d279c,apnews.com,EXPLAINER: Is COVID-19 winding down? Scientist...,Is the coronavirus on its way out? You might t...,2022-09-07T00:30:51.325Z
5059,6317e258adff9198f5a06995,bitcoinmagazine.com,How Bitcoin Educates The World About Finance,Bitcoin brings financial education to the fore...,2022-09-07T00:09:33.167Z
5060,631743e12107b12c267b21dc,coindesk.com,Crypto Hedge Fund Three Arrows Capital Wallet ...,The removal of liquidity was the first activit...,2022-09-06T12:58:08.319Z


In [9]:
def combine_news_text(example):
  title=example.get("title","") or ""
  description=example.get("description","") or ""

  return {"full_text": f"Title:{title}\n Description:{description}"}

In [10]:
news_dataset=news_dataset.map(combine_news_text)

Map:   0%|          | 0/5062 [00:00<?, ? examples/s]

In [11]:
news_dataset

Dataset({
    features: ['_id', 'main_domain', 'title', 'description', 'created_at', 'full_text'],
    num_rows: 5062
})

In [12]:
news_dataset[0]

{'_id': '6453d70d358e80adbfc4cb2b',
 'main_domain': 'cnbc.com',
 'title': 'Dow drops 400 points, turns negative for the year as bank fears grow: Live updates',
 'description': 'Regional banks led the broader market lower as contagion fears resurfaced.',
 'created_at': '2023-05-04T16:01:46.448Z',
 'full_text': 'Title:Dow drops 400 points, turns negative for the year as bank fears grow: Live updates\n Description:Regional banks led the broader market lower as contagion fears resurfaced.'}

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig # Add BitsAndBytesConfig

model_id="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
print(f"loading model:{model_id}")

# Define quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True
)

model=AutoModelForCausalLM.from_pretrained(model_id,
                                           torch_dtype="auto",
                                           device_map="auto",
                                           quantization_config=quantization_config, # Pass the quantization config
                                           trust_remote_code=True)
model.eval()
print("model loaded successfully in 4 bit")

loading model:deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

model loaded successfully in 4 bit


In [14]:
tokenizer=AutoTokenizer.from_pretrained(model_id,trust_remote_code=True)
print("Tokeniser loaded successfully")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokeniser loaded successfully


In [15]:
llm_pipeline=pipeline("text-generation",
                      model=model,
                      tokenizer=tokenizer,
                      torc_dtype="auto",
                      device_map="auto")
print("Text generation pipeline created successfully")

Text generation pipeline created successfully


In [16]:
test_question="Explain what electric cars are in somple terms. Keep thinking short"
test_prompt=f"<|im_start|>user\n{test_question}<|im_end|>\n<|im_start|>assistant\n<think>\n"
print(f"Testprompt:\n{test_prompt}")

Testprompt:
<|im_start|>user
Explain what electric cars are in somple terms. Keep thinking short<|im_end|>
<|im_start|>assistant
<think>



In [19]:
llm_pipeline=pipeline("text-generation",
                      model=model,
                      tokenizer=tokenizer,
                      torch_dtype="auto", # Corrected typo from torc_dtype to torch_dtype
                      device_map="auto")
print("Text generation pipeline created successfully")

print("testing with sample prompt")
outputs=llm_pipeline(test_prompt,
                     max_new_tokens=4000,
                     do_sample=True,
                     temperature=0.7,
                     top_p=0.9)
outputs

`torch_dtype` is deprecated! Use `dtype` instead!
Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Text generation pipeline created successfully
testing with sample prompt


[{'generated_text': "<|im_start|>user\nExplain what electric cars are in somple terms. Keep thinking short<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, so I'm trying to understand what electric cars are in simple terms. I've heard people talk about electric cars before, but I'm not entirely sure what they are. Let me think through this step by step.\n\nFirst, I know that electric cars are related to cars, so they probably have some of the same features. Maybe they have the same kind of engine, like a gasoline engine, but instead of those, they use something called an electric motor. That makes sense because electric cars would be powered by electricity, not the same kind of fuel.\n\nI remember seeing that electric cars are often named after famous people, like Tesla, Toyota, or Ford. These companies probably make electric cars to showcase their technology. Maybe Tesla makes electric cars like the Model 3, and Toyota might make electric cars like the Model S. These names suggest tha

In [20]:
def format_model_output(output):

  reason_start=output.find("<think>")+len("<think>")
  reason_end=output.find("</think>")
  reason=output[reason_start:reason_end].strip()

  output_start=reason_end+len("</think>")
  model_output_content=output[output_start:].strip()

  reason=f"**Reason**:\n{reason}\n"
  output=f"**Output**:\n{model_output_content}\n"

  return reason,output


In [21]:
full_output=outputs[0]["generated_text"]
reason,output=format_model_output(full_output)
print_markdown(reason+output)

**Reason**:
Okay, so I'm trying to understand what electric cars are in simple terms. I've heard people talk about electric cars before, but I'm not entirely sure what they are. Let me think through this step by step.

First, I know that electric cars are related to cars, so they probably have some of the same features. Maybe they have the same kind of engine, like a gasoline engine, but instead of those, they use something called an electric motor. That makes sense because electric cars would be powered by electricity, not the same kind of fuel.

I remember seeing that electric cars are often named after famous people, like Tesla, Toyota, or Ford. These companies probably make electric cars to showcase their technology. Maybe Tesla makes electric cars like the Model 3, and Toyota might make electric cars like the Model S. These names suggest that the cars are designed to look and perform like the famous models.

Another thing I'm thinking about is the range. Electric cars have a range, which is how far they can go on a single charge of electricity. For example, Tesla's Model 3 can go about 270 miles on a single charge, which is pretty far. This range is important because it affects how much you can drive with a single charge of power. It also affects how much you need to charge the battery to go further.

I'm also curious about how electric cars work. They probably have a battery that stores energy, and the electric motor uses that energy to move the car. So, just like how a battery-powered car works, an electric car would have an electric motor and a battery. The motor would drive the wheels, allowing the car to move forward.

I wonder if electric cars can go very far, like 200 miles or more, or if they have limits on how far they can go. I think the range can vary from car to car, but generally, it's something that's set before you start driving. So, maybe some electric cars have a longer range and go farther than others.

I'm also thinking about how electric cars compare to other types of cars. Maybe they're different in some ways, like how they look, how they're designed, or how they operate. For example, some cars might be more expensive, or they might have different safety features. But in terms of electric cars, I think they share some similarities with gasoline cars, like being powered by electricity and using the same kind of engine.

Another thought is about charging. Electric cars probably need to be charged before you can drive them. So, you have to charge the battery before you start driving, and you can only drive so far before you have to charge again. This charging process might take time and energy, so it's important to have an efficient charging system.

I'm also curious about the safety features of electric cars. I think they have different safety features compared to gasoline cars, like better emergency braking or more advanced safety systems. This makes sense because electric cars rely more on electricity, so they need to be safer in some way.

In summary, electric cars are powered by electricity, have similar features to gasoline cars, like engines and wheels, but they have a range that allows them to travel a certain distance. They use electric motors to move the car, and they have battery systems to store energy. Electric cars can vary in range, cost, and design, but they share some similarities with gasoline cars and have different safety features.

I think I've got a better understanding now. Electric cars are powered by electricity, have engines and wheels similar to gasoline cars, and have a range that allows them to travel a certain distance. They use electric motors and have battery systems. The range can vary, and electric cars can have different designs and safety features compared to gasoline cars.
**Output**:
**Understanding Electric Cars**

1. **Power Source**: Electric cars are powered by electricity, using an electric motor instead of a gasoline engine. This power is provided by a battery system.

2. **Design**: These cars often resemble famous models like the Model S or the Tesla Model 3, showcasing design and performance.

3. **Range**: Electric cars have a range, or distance, they can travel on a single charge of electricity. For example, Tesla's Model 3 can travel about 270 miles.

4. **Electrical System**: The car features an electric motor and a battery system. The motor drives the wheels, allowing movement forward.

5. **Range Limitations**: The range of electric cars can vary, with some offering longer distances like 200 miles.

6. **Charging Process**: Electric cars need charging before driving, typically at a battery with a fast charge capability.

7. **Safety Features**: Electric cars may have safety features, like emergency braking, compared to gasoline cars.

8. **Design Variations**: Electric cars can have different designs and safety features compared to gasoline cars, offering various performance options.

In summary, electric cars are powered by electricity, share features with gasoline cars, have ranges allowing travel, and use electric motors. They have battery systems, vary in range, design, and safety features, and require charging before use.


In [22]:
import re
import random

def get_random_news():
  # Check if news_dataset is defined in the global scope
  if 'news_dataset' not in globals():
    return "Error: news_dataset not initialized. Please ensure the dataset loading cells are run."

  # Now that we know news_dataset exists, we can safely check if it's empty
  if not news_dataset or len(news_dataset) == 0:
    return "Error: The news_dataset is empty. Cannot fetch random news."

  random_index = random.randint(0, len(news_dataset) - 1)
  news_text = news_dataset[random_index]["full_text"]
  print("fetched news article.")
  return news_text

In [ ]:
import gradio as gr

def get_random_news_gradio():
  """Fetches a random news article and clears the analysis output."""
  news = get_random_news() # Uses the get_random_news from cell Xo0qEOfu2-IS
  return news, ""

def analyze_news_gradio(news_input_text):
  """Analyzes the provided news text using the LLM pipeline."""
  if not news_input_text or news_input_text.startswith("Error"): # Check for error from get_random_news
    return "Please fetch a news article first or check for errors in fetching."

  print(f"Analyzing news: {news_input_text[:100]}...")

  # Construct the prompt for the LLM, encouraging a structured thought process.
  prompt = f"<|im_start|>user\nAnalyze the sentiment (positive, negative, neutral, or mixed) and identify key entities (like company names, people, financial terms, or events) in the following financial news article. Provide your reasoning before the final analysis.\n\nNews Article: {news_input_text}\n<|im_end|>\n<|im_start|>assistant\n<think>I need to carefully read the news article to identify its main theme, extract any significant entities like company names, people, or financial terms, and then determine the overall sentiment expressed (e.g., positive, negative, neutral, mixed).</think>\n"

  # Generate response from the LLM
  outputs = llm_pipeline(prompt,
                         max_new_tokens=700, # Increased tokens for a more detailed analysis
                         do_sample=True,
                         temperature=0.7,
                         top_p=0.9)
  full_output = outputs[0]["generated_text"]

  # Extract reason and output from the generated text
  # format_model_output is defined in cell SPnY2Erlz0Ff
  reason, model_output = format_model_output(full_output)
  return reason + model_output


with gr.Blocks() as demo:
    gr.Markdown("# Financial News Sentiment and Entity Analyzer")
    gr.Markdown("Get a random financial news article and analyze its sentiment and key entities using the DeepSeek-R1-Distill-Qwen-1.5B model.")

    with gr.Row():
        with gr.Column():
            news_input = gr.Textbox(label="Financial News Article", lines=15, interactive=True, placeholder="Click 'Get Random News' to load an article.")
            get_news_btn = gr.Button("Get Random News")
        with gr.Column():
            analysis_output = gr.Markdown(label="Analysis Result", value="")
            analyze_btn = gr.Button("Analyze News")

    get_news_btn.click(
        fn=get_random_news_gradio,
        inputs=[],
        outputs=[news_input, analysis_output]
    )

    analyze_btn.click(
        fn=analyze_news_gradio,
        inputs=[news_input],
        outputs=analysis_output
    )

demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b45f74bf818f7f311d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


fetched news article.


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Analyzing news: Title:Cleopatra Testing Poisons 🐍 Elena Bondar: Oil-on-Canvas
 Description:Queen Cleopatra looks on ...
fetched news article.


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Analyzing news: Title:Oil prices edge higher despite the West's Russian price cap talks and more Chinese lockdowns
 ...
fetched news article.


Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Analyzing news: Title:Analysis: Watch out risk assets, the rout in bonds is coming your way
 Description:The selling...
